# SiegeScout — minimap YOLO-nano training

Trains YOLO11-nano on your **minimapVision** Roboflow export (CC BY 4.0) and
exports `minimap.onnx` for the bot's local CPU inference path
(`local_inference.OnnxDetector`).

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**You need:** the Roboflow YOLO export zip — currently
`minimapVision 2.v3i.yolov12.zip` in the project root (76 images, 800×600).
Any export with the same 13-class schema works; upload it when cell 3 asks.

Full run ≈ 25–40 min on a T4: install → upload → remap 13→9 classes +
cut 240×240 tiles → train → validate → export ONNX → sanity check → download.


In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()


In [ ]:
# Upload the Roboflow export zip and unpack it.
import os
import zipfile

from google.colab import files

RAW_DIR = "/content/dataset_raw"
if not os.path.isdir(RAW_DIR):
    uploaded = files.upload()  # pick the minimapVision export zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(RAW_DIR)
# Some zips nest everything one folder deep - normalize.
if not os.path.isdir(os.path.join(RAW_DIR, "train")):
    inner = [d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))]
    assert len(inner) == 1, f"unexpected zip layout: {inner}"
    RAW_DIR = os.path.join(RAW_DIR, inner[0])
print("dataset at", RAW_DIR, "->", sorted(os.listdir(RAW_DIR)))


In [ ]:
# Remap 13 raw classes -> the 9 with enough annotations, then TILE.
# Minimap objects are ~5x7 px in these 800x600 captures: letterboxing the
# whole image to the model input shrinks them to 2-3 px (invisible - the
# first training run scored mAP50 0.08). Cutting 240x240 tiles instead
# keeps them native-size, and 240 matches the bot's ~235x230 minimap crop,
# so training scale == inference scale.
import shutil
from collections import Counter
from pathlib import Path

import cv2

# The 13-class schema shared by the minimapVision exports, in data.yaml
# order — verify against data.yaml if you re-export from Roboflow.
RAW_NAMES = ["dungeon", "enter", "exit", "itemAngel", "itemBlue", "itemSatan",
             "oreBlue", "oreCopper", "oreGold", "oreGreen", "oreRuby",
             "player", "portal"]
# The dropped four (itemAngel 2, itemSatan 6, oreBlue 2, oreCopper 15
# annotations TOTAL in the current export) become background until more
# labels exist.
# KEEP order = the model's class ids - MUST match
# LOCAL_MINIMAP_MODEL_CLASSES in config.py.
KEEP_CLASSES = ["dungeon", "enter", "exit", "itemBlue",
                "oreGold", "oreGreen", "oreRuby", "player", "portal"]

TILE = 240        # match the bot's minimap crop size
GRID = (4, 3)     # 4x3 overlapping tiles cover 800x600
KEEP_FRAC = 0.60  # keep a box if >=60% of its area lands inside the tile
EMPTY_KEEP = 4    # keep 1 in 4 object-free tiles as pure background

remap = {RAW_NAMES.index(n): i for i, n in enumerate(KEEP_CLASSES)}
DATA_DIR = Path("/content/dataset")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)


def tile_origins(full, n):
    if n == 1:
        return [0]
    stride = (full - TILE) / (n - 1)
    return [round(i * stride) for i in range(n)]


for split in ("train", "valid", "test"):
    img_out = DATA_DIR / split / "images"
    lbl_out = DATA_DIR / split / "labels"
    img_out.mkdir(parents=True)
    lbl_out.mkdir(parents=True)
    counts = Counter()
    n_tiles = n_empty = 0
    for img_path in sorted((Path(RAW_DIR) / split / "images").glob("*.jpg")):
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        boxes = []  # (new_cid, cx, cy, w, h) in absolute pixels
        lbl_path = Path(RAW_DIR) / split / "labels" / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                p = line.split()
                if len(p) == 5 and int(p[0]) in remap:
                    boxes.append((remap[int(p[0])], float(p[1]) * W,
                                  float(p[2]) * H, float(p[3]) * W,
                                  float(p[4]) * H))
        for ty in tile_origins(H, GRID[1]):
            for tx in tile_origins(W, GRID[0]):
                kept = []
                for cid, cx, cy, w, h in boxes:
                    x1 = max(cx - w / 2, tx); y1 = max(cy - h / 2, ty)
                    x2 = min(cx + w / 2, tx + TILE); y2 = min(cy + h / 2, ty + TILE)
                    if x2 - x1 <= 1 or y2 - y1 <= 1:
                        continue
                    if (x2 - x1) * (y2 - y1) < KEEP_FRAC * w * h:
                        continue  # mostly outside: the neighbor tile owns it
                    kept.append((cid, (x1 + x2) / 2 - tx, (y1 + y2) / 2 - ty,
                                 x2 - x1, y2 - y1))
                if not kept:
                    n_empty += 1
                    if n_empty % EMPTY_KEEP:
                        continue
                name = f"{img_path.stem}_t{tx}x{ty}"
                cv2.imwrite(str(img_out / f"{name}.jpg"),
                            img[ty:ty + TILE, tx:tx + TILE])
                (lbl_out / f"{name}.txt").write_text("".join(
                    f"{cid} {cx / TILE:.6f} {cy / TILE:.6f} "
                    f"{w / TILE:.6f} {h / TILE:.6f}\n"
                    for cid, cx, cy, w, h in kept))
                n_tiles += 1
                for k in kept:
                    counts[KEEP_CLASSES[k[0]]] += 1
    print(f"{split}: {n_tiles} tiles, "
          f"{dict(sorted(counts.items(), key=lambda kv: -kv[1]))}")

(DATA_DIR / "data.yaml").write_text(
    f"path: {DATA_DIR}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    f"nc: {len(KEEP_CLASSES)}\n"
    f"names: {KEEP_CLASSES}\n")
print((DATA_DIR / "data.yaml").read_text())


In [ ]:
# Train. ~20-30 min on a T4.
from ultralytics import YOLO

# Static export size - must match LOCAL_MINIMAP_MODEL_INPUT_SIZE. 240px
# tiles at 480 input = 2x upscale, so ~5x7 px objects train at ~11x14 px.
IMGSZ = 480

model = YOLO("yolo11n.pt")  # nano - smallest, fastest on the bot's CPU
model.train(
    data=str(DATA_DIR / "data.yaml"),
    imgsz=IMGSZ,
    epochs=150,
    patience=40,
    batch=16,
    seed=0,
    mosaic=0.0,   # mosaic quarters each image - lethal for tiny objects
    scale=0.3,    # mild zoom jitter only, for the same reason
    degrees=0.0,  # the minimap never rotates
    hsv_h=0.01,   # ore classes are told apart by HUE - keep hue jitter tiny
    plots=True,
)
BEST = str(model.trainer.best)
print("best weights:", BEST)


In [ ]:
# Per-class validation. This is a training sanity check only - the real
# acceptance gate is `python -m eval.score --detector all` on labeled
# captures from the actual bot minimap.
from ultralytics import YOLO

best = YOLO(BEST)
metrics = best.val(data=str(DATA_DIR / "data.yaml"), imgsz=IMGSZ)
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    print(f"{KEEP_CLASSES[int(cls_idx)]:<10} AP50={metrics.box.ap50[i]:.3f}")
print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")


In [ ]:
# Export ONNX: opset 12, static input, no in-graph NMS (the bot does
# numpy NMS itself - see local_inference.py).
%pip install -q onnx onnxslim
import shutil

import onnx

onnx_path = best.export(format="onnx", opset=12, imgsz=IMGSZ,
                        dynamic=False, simplify=True)
shutil.copy(onnx_path, "/content/minimap.onnx")

m = onnx.load("/content/minimap.onnx")
def dims(vi):
    return [d.dim_value for d in vi.type.tensor_type.shape.dim]
print("input :", [(i.name, dims(i)) for i in m.graph.input])
print("output:", [(o.name, dims(o)) for o in m.graph.output])
# Expect: input images [1, 3, 480, 480]; output [1, 13, 4725]
# (13 = 4 box coords + 9 classes; 4725 = anchors at 480px).


In [ ]:
# Sanity check OUTSIDE ultralytics: onnxruntime + the same numpy
# letterbox/decode the bot uses. Boxes should land on minimap objects.
# (No NMS here, so duplicate boxes per object are expected and fine.)
import glob
import random

import cv2
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession("/content/minimap.onnx",
                            providers=["CPUExecutionProvider"])
img = cv2.imread(random.choice(glob.glob(str(DATA_DIR / "valid/images/*"))))

h, w = img.shape[:2]
scale = min(IMGSZ / w, IMGSZ / h)
nw, nh = round(w * scale), round(h * scale)
canvas = np.full((IMGSZ, IMGSZ, 3), 114, np.uint8)
px, py = (IMGSZ - nw) // 2, (IMGSZ - nh) // 2
canvas[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh))
blob = (cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)[None]
        .astype(np.float32) / 255.0)

out = np.squeeze(sess.run(None, {"images": blob})[0]).T  # (N, 4+nc)
conf = out[:, 4:].max(axis=1)
for row in out[conf >= 0.4]:
    cx, cy, bw, bh = row[:4]
    cls = KEEP_CLASSES[int(row[4:].argmax())]
    x1, y1 = int((cx - bw / 2 - px) / scale), int((cy - bh / 2 - py) / scale)
    x2, y2 = int((cx + bw / 2 - px) / scale), int((cy + bh / 2 - py) / scale)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)
    cv2.putText(img, cls, (x1, max(10, y1 - 3)),
                cv2.FONT_HERSHEY_PLAIN, 0.9, (0, 255, 0))

from google.colab.patches import cv2_imshow
cv2_imshow(img)


In [ ]:
from google.colab import files
files.download("/content/minimap.onnx")
print("""
Next steps on the dev machine:
 1. Copy minimap.onnx to assets/models/minimap.onnx
 2. Score it against real captures (needs labeled eval frames):
      python -m eval.score --detector all
 3. Flip LOCAL_MINIMAP_MODEL_ENABLED=True in config.py only if it beats the
    heuristic baseline (+10 F1 acceptance gate, see the CV spec).
Caveat: the bot's small minimap (~235x230) is smaller than these 800x600
training crops - the eval score IS the transfer test, not the mAP above.
""")
